# Data Generation / Baselining

In [ ]:
# Shared functions/imports that you'll need
import json
import random
from tqdm import tqdm
from llmclient import LLMClient, get_llm_response, retrieve
from data_prompts import generate_datapoint_from_datapoint_per_criterion

random.seed(123)


# To tell the models how to write
locales = {
    "West Frisian": "West Frisian",
    "Cornish": "West Country (Cornish) English",
    "Geordie": "Geordie English",
    "AAVE": "African-American Vernacular English",
    "Yorkshire": "Yorkshire English",
    "Welsh": "Welsh English"
}

locale_codes = {
    "AAVE": "en-US-AAVE",
    "West Frisian": "fy-NL", 
    "Geordie": "en-UK-Geo",
    "Yorkshire": "en-UK-York",
    "Cornish": "en-UK-Wes",
}


# Behaviourally-Aligned Data

This is synthetic data that is either human-corrected and annotated, or LLM-aligned annotated/judged (depending on the experiment, this script supports both)

In [ ]:
from data_prompts import get_transliteration_prompt, get_generation_prompt
from evaluation_prompts import get_evaluator_prompt_single_criteria

is_alignment = True # Judged by LLM (true) or human (false)

# Only use high-quality models -- here's where the aligned judge goes.
# You need to change this depending on locale.
models_available = [("gpt-41-longco-2025-04-14", {"max_tokens": 6000}),
                    ("anthropic-claude-opus-4-1", {"max_tokens": 6000}),
                    ]

# We don't finetune Qwen3 on its own predictions.
bad_models = [("gpt-5-mini", {"max_completion_tokens": 6000}, False),
              ("phi-4", {"max_tokens": 6000}),
              ]

prompt_model = LLMClient({"max_completion_tokens": 6000}, "gpt-41-longco-2025-04-14")
fallback_model = ("gpt-41-longco-2025-04-14", {"max_tokens": 6000})


def collect_data(source):
    lines = []
    if source == "OpenCode":
        from datasets import load_dataset
        ds = load_dataset("nvidia/OpenCodeReasoning", "split_0")
        lines = [{"en-US-std Prompt": l["input"]} for l in ds['split_0']]
    elif source == "GSM8K":
        lines = [{"en-US-std Prompt": json.loads(l)["question"]} for l in open("../Data/gsm8k_train.jsonl", "r", encoding="utf-8")]
    elif source == "OpenOrca":
        ...
    elif source == "WildChat":
        ...
    elif source == "shp":
        ds = load_dataset("stanfordnlp/SHP")
    return lines


def assert_gen_fn(x):
    if "response" not in x: return False
    if type(x["response"]) != str: return False
    return True


# Load a seed dataset
data = ... 
ofile = ...


In [ ]:
for code, nat_str in locales.items():

    for ix in tqdm(range(len(data)), desc=code):

        prompt = data[ix]["en-US-std Prompt"]
        new_entry = {
            "Index": ix,
            "Locale": code,
            "Source": data[ix]["Source"],
            "en-US-std Prompt": prompt,
        }

        name, params = random.choice(models_available)
        llm = LLMClient(params, name)
        # Generate the transliterated prompt
        gen_llm = prompt_model
        transliterated_prompt, failed_state = retrieve(get_transliteration_prompt(prompt, locale=nat_str),
                                                       prompt_model, assert_fn=lambda x: "transliteration" in x,
                                                       DEFAULT_RESPONSE={"transliteration": prompt}, max_tries=3)
        if failed_state: 
            name, params = fallback_model
            gen_llm = LLMClient(params, name)
            print(f"Warn, transliteration for {ix} {code} failed, falling back!")
            transliterated_prompt, failed_state = retrieve(get_transliteration_prompt(prompt, locale=nat_str), 
                                                           fallback_model, assert_fn=lambda x: "transliteration" in x,
                                                           DEFAULT_RESPONSE={"transliteration": prompt}, max_tries=3)
        if failed_state:
            print(f"Warn, transliteration for {ix} {code} failed! Skipping")
            continue

        new_entry["Metadata"] = {
            "PromptModel": gen_llm.model_name,
            "PromptFailedState": failed_state
        }
        new_entry["Prompt"] = transliterated_prompt["transliteration"]
        if type(transliterated_prompt["transliteration"]) != str: continue

        # Get the LLM response
        generation_prompt = get_generation_prompt(transliterated_prompt["transliteration"], locale=nat_str)
        resp, failed_state = retrieve(generation_prompt, llm, assert_fn=assert_gen_fn)
        if failed_state:
            print(f"Warn, generation for {ix} {code} ({llm.model_name}) failed, falling back")
            name, params = fallback_model
            llm = LLMClient(params, name)
            resp, failed_state = retrieve(generation_prompt, llm, assert_fn=assert_gen_fn, DEFAULT_RESPONSE={"response": "FAIL"})

        if failed_state:
            print(f"Warn, response for {ix} {code} failed! Skipping")
            continue

        new_entry["Response"] = resp["response"]
        new_entry["Metadata"]["ResponseModel"] = llm.model_name

        # Create a bad output with a bad model.
        if is_alignment:
            model_name, params = random.choice(bad_models)
            resp, failed_state = retrieve(generation_prompt, llm, assert_fn=assert_gen_fn, DEFAULT_RESPONSE={"response": "FAIL"})
            if failed_state:
                continue
            new_entry["NegResponse"] = resp["response"]
            new_entry["Metadata"]["NegResponseModel"] = model_name

        # Judge this one -- this is only done when you have an aligned judge (i.e., synthetic data gen)
        if is_alignment:
            for crit in ["c1", "c2a", "c2b", "c3", "c4", "c5"]:
                prompt = get_evaluator_prompt_single_criteria({
                                                            "Prompt": new_entry["Prompt"],
                                                            "Output": new_entry["Response"]
                                                            },  criterion=crit, num_exemplars=0,
                                                            locale=nat_str, exemplar_dataset=[], request_reasons=False)
                response, failure = retrieve(prompt, llm,  assert_fn = lambda x: crit in x, 
                                                DEFAULT_RESPONSE={crit: 0}, max_tries=3)
                if response[crit] == 0:
                    print("Judgement failed")
                    continue

        with open(ofile, "a", encoding="utf-8") as f:
            f.write(json.dumps(new_entry, ensure_ascii=False) + "\n")



In [ ]:
for code, nat_str in locales.items():
    lines = [json.loads(l) for l in open(ofile, "r", encoding="utf-8")]
    with open(ofile, "w", encoding="utf-8") as f:
        json.dump(lines, f, ensure_ascii=False)


# ICL Baselining

Establish baselines for every LLM (i.e., predictions per shot per prompt)

In [ ]:
# Shared functions/imports that you'll need
import json
import random
from tqdm import tqdm
from llmclient import LLMClient, get_llm_response, retrieve


from evaluation_prompts import (get_evaluator_prompt_single_criteria,
                                get_evaluator_prompt_all_criteria,
                                )
from shared_prompt_utils import ALL_CRIT_DEFAULT_RESPONSE

random.seed(123)

# To tell the models how to write
locales = {
    "AAVE": "African-American Vernacular English",
    "West_Frisian": "West Frisian",
    "Geordie": "Geordie English",
    "Yorkshire": "Yorkshire English",
    "Cornish": "West Country (Cornish) English",
}

locale_codes = {
    "AAVE": "en-US-AAVE",
    "WestFrisian": "fy-NL",
    "Geordie": "en-UK-Geo",
    "Yorkshire": "en-UK-York",
    "Cornish": "en-UK-Wes",
}

dialect_file = "West_Frisian"    # Note that for fy-NL it is not a dialect
locale = locales[dialect_file]
root = f"icl_predictions/{dialect_file}/"

dataset = json.load(open(f"data/human_annotated/{dialect_file.lower()}_dataset_with_rubric_shuffled.json", "r", encoding="utf-8")) #nl stands for "natural-language", not the language code for Dutch (nl-NL)

In [ ]:
models_available = [("dev-gpt-5-mini", {"max_completion_tokens": 6000}),
                    ("dev-gpt-41-longco-2025-04-14", {"max_tokens": 256, "temperature": 0}),
                    ("dev-phi-4", {"max_tokens": 256}),
                    ("dev-anthropic-claude-opus-4-1", {"max_tokens": 6000, "temperature": 0}),
                    ("dev-qwen-3-8b", {"max_tokens": 6000, "temperature": 0}),
                    ]
num_exemplars = [0, 5, 20] # We did 40 for some but it's just too expensive

## All Criteria

In [ ]:
# All criteria
def assert_fn_w_reasons(x):
    return any([k in x for k in [
        "c1", "c1_reason", "c2a", "c2a_reason", "c2b", "c2b_reason", 
        "c3", "c3_reason", "c4", "c4_reason", "c5", "c5_reason"
    ]])

def assert_fn_no_reasons(x):
    return any([k in x for k in ["c1", "c2a", "c2b", "c3", "c4", "c5",]])


def baseline_all_criteria(models, num_exemplars):
    for MODEL, params in models:
        llm = LLMClient(params, MODEL)

        for ex in num_exemplars:
            for ix in tqdm(range(len(dataset)), desc=MODEL):

                entry = dataset[ix]
                pred_entry = {k:v for k, v in entry.items()}

                # No breakdown
                prompt = get_evaluator_prompt_all_criteria(entry, num_exemplars=ex, exemplar_dataset=dataset, request_breakdown=False, 
                                                           locale=locale, request_reasons=False)
                response, failure = retrieve(prompt, llm, assert_fn=lambda x: "Label" in x, max_tries=3)
                pred_entry["NoBreakdown"] = {"response": response, "failed": failure}

                # Breakdown no reasons
                prompt = get_evaluator_prompt_all_criteria(entry, num_exemplars=ex, exemplar_dataset=dataset, request_breakdown=True, 
                                                           locale=locale, request_reasons=False)
                response, failure = retrieve(prompt, llm, assert_fn=assert_fn_no_reasons, 
                                             DEFAULT_RESPONSE={"c1": 0, "c2a": 0, "c2b": 0, "c3": 0, "c4": 0, "c5": 0}, max_tries=3)
                pred_entry["BreakdownNoReasons"] = {"response": response, "failed": failure}

                # # Breakdown + reasons
                prompt = get_evaluator_prompt_all_criteria(entry, num_exemplars=ex, exemplar_dataset=dataset, request_breakdown=True, 
                                                           locale=locale, request_reasons=True)
                response, failure = retrieve(prompt, llm, assert_fn=assert_fn_w_reasons, DEFAULT_RESPONSE=ALL_CRIT_DEFAULT_RESPONSE, max_tries=3)
                pred_entry["BreakdownReasons"] = {"response": response, "failed": failure}

                with open(f'{root}/{MODEL}_{ex}_all_crit.json', "a", encoding="utf-8") as f:
                    f.write(json.dumps(pred_entry, ensure_ascii=False) + "\n")

baseline_all_criteria(models_available, num_exemplars)

## Per criterion

In [ ]:
# Per-criterion
def baseline_per_criteria(models, num_exemplars):

    for MODEL, params in models:
        llm = LLMClient(params, MODEL)

        for ex in num_exemplars:
            
            labels, labels_reasons, true_labels = {}, {}, {}
            for ix in tqdm(range(len(dataset)), desc=MODEL):

                entry = dataset[ix]
                pred_entry = {k:v for k, v in entry.items()}

                pred_entry["NoReasons"] = {}
                for crit in ["c1", "c2a", "c2b", "c3", "c4", "c5"]:
                    prompt = get_evaluator_prompt_single_criteria(entry, criterion=crit, num_exemplars=ex, 
                                                                  locale=locale, exemplar_dataset=dataset, request_reasons=False)
                    response, failure = retrieve(prompt, llm,  assert_fn = lambda x: crit in x, 
                                                 DEFAULT_RESPONSE={crit: 0}, max_tries=3)
                    pred_entry["NoReasons"][crit] = {"response": response[crit], "failure": failure}

                    if crit not in labels: labels[crit] = []
                    labels[crit].append(pred_entry["NoReasons"][crit])
                    if crit not in true_labels: true_labels[crit] = []
                    true_labels[crit].append(entry["Rubric"][crit])

                pred_entry["Reasons"] = {}
                for crit in ["c1", "c2a", "c2b", "c3", "c4", "c5"]:
                    # Breakdown no reasons
                    prompt = get_evaluator_prompt_single_criteria(entry, criterion=crit, num_exemplars=ex, 
                                                                  locale=locale, exemplar_dataset=dataset, request_reasons=True)
                    response, failure = retrieve(prompt, llm, assert_fn = lambda x: crit in x, 
                                                 DEFAULT_RESPONSE={crit: 0, crit + "_reason": "FAIL"}, max_tries=3)
                    pred_entry["Reasons"][crit] = {"response": response[crit], "failure": failure}

                    if crit not in labels_reasons: labels_reasons[crit] = []
                    labels_reasons[crit].append(pred_entry["Reasons"][crit])

                with open(f'{root}/{MODEL}_{ex}_per_crit.json', "a", encoding="utf-8") as f:
                    f.write(json.dumps(pred_entry, ensure_ascii=False) + "\n")

baseline_per_criteria(models_available, num_exemplars)